In [1]:
import re
import json
from functools import reduce
from copy import deepcopy
from itertools import product
import pandas as pd

In [2]:

def make_anti(particle: str) -> str:
    #print(particle)
    wronf_p = {"anti-Lambda_c-": "Lambda_c+", "Lambda_c(2593)+":"anti-Lambda_c(2593)-", "Lambda_c(2625)+":"anti-Lambda_c(2625)-"}
    if particle in wronf_p:
        return wronf_p[particle]
    if particle in ["pi0", "rho0", "K_S0"]:
        return particle
    if particle.endswith("+"):
        return particle[:-1] + "-"
    if particle.endswith("-"):
        return particle[:-1] + "+"
    if "0" in particle:
        if particle.startswith("anti-"):
            return particle[5:]
        return "anti-" + particle
    return particle


In [3]:
print(make_anti("anti-D0"))

D0


In [4]:
def parse_decay_block(text: str) -> dict[str, list[dict]]:
    """
    Парсит текст в формате DECAY BLOCK в словарь с продуктами, BR и моделью.
    Также обрабатывает директиву CDecay для автогенерации сопряжённых распадов.
    """
    decay_dict = {}
    current_decay = None
    cdecay_links = []

    MODEL_KEYWORDS = {
        'PHOTOS', 'ISGW2', 'PHSP', 'SVS', 'STS', 'PYTHIA',
        'TAULNUNU', 'TAUSCALARNU', 'TAUVECTORNU',
        'SVV_HELAMP', 'VSP_PWAVE'
    }

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        if line.startswith("Decay"):
            parts = line.split()
            if len(parts) > 1:
                current_decay = parts[1]
                decay_dict[current_decay] = []
            continue

        if line.startswith("CDecay"):
            parts = line.split()
            if len(parts) > 1:
                cdecay_links.append(parts[1])
            continue

        if line.startswith("Enddecay"):
            current_decay = None
            continue

        if current_decay:
            line = re.sub(r'#.*', '', line)
            line = line.rstrip(';')
            tokens = line.split()

            if len(tokens) < 2:
                continue

            try:
                br = float(tokens[0])
            except ValueError:
                continue

            model_start = next(
                (i for i, t in enumerate(tokens[1:], 1)
                 if re.fullmatch(r'[A-Z0-9_]+', t) and t in MODEL_KEYWORDS),
                len(tokens)
            )

            products = tokens[1:model_start]
            model = ' '.join(tokens[model_start:])

            decay_dict[current_decay].append({
                'branching_ratio': br,
                'products': products,
                'model': model
            })

    for anti_particle in cdecay_links:

        orig_particle = make_anti(anti_particle)


        anti_decays = []
        for decay in decay_dict[orig_particle]:
            anti_products = [make_anti(p) for p in decay["products"]]
            anti_decays.append({
                'branching_ratio': decay['branching_ratio'],
                'products': anti_products,
                'model': decay['model']
            })
        decay_dict[anti_particle] = anti_decays

    return decay_dict


In [5]:

text_data2 = str()
with open("DECAY_1.DEC", "r") as outfile:
    text_data2 = reduce(lambda x, y: x +y, [i for i in outfile.readlines()])

In [6]:
text_data1 = str()
with open("my_dec.DEC", "r") as outfile:
    text_data1 = reduce(lambda x, y: x +y, [i for i in outfile.readlines()])


In [23]:
decays1 = parse_decay_block(text_data1)
decays2 = parse_decay_block(text_data2)


In [25]:
def find_final_states(particle: str, decay_dict: dict) ->   list[list[str]]:

    if particle not in decay_dict:
        return [[particle]]

    final_states = []

    for decay in decay_dict[particle]:
        branches = [find_final_states(p, decay_dict) for p in decay["products"]]
        from itertools import product
        for combination in product(*branches):
            flat_state = []
            for group in combination:
                flat_state.extend(group)
            final_states.append(flat_state)
    for final in final_states:
        final = final.sort()
    return final_states


In [27]:
def prune_unreachable_particles(decay_dict: dict, root: str) -> dict:
    reachable = set()
    frontier = {root}

    while frontier:
        next_frontier = set()
        for particle in frontier:
            if particle not in decay_dict:
                continue
            reachable.add(particle)
            for decay in decay_dict[particle]:
                for p in decay["products"]:
                    if p not in reachable:
                        next_frontier.add(p)
        frontier = next_frontier

    pruned_dict = {p: d for p, d in decay_dict.items() if p in reachable}
    return pruned_dict


In [28]:
decays21 = prune_unreachable_particles(decays2, "B_s0")
decays22 = prune_unreachable_particles(decays2, "anti-B_s0")

In [29]:
def merge_decay_dicts(*dicts: dict) -> dict:
    merged = {}
    for d in dicts:
        for particle, decays in d.items():
            if particle not in merged:
                merged[particle] = []
            merged[particle].extend(decays)

    # Удалим дубликаты по значению
    for particle in merged:
        merged[particle] = list(set(merged[particle]))

    return merged


In [ ]:
def extract_all_products(decay_dict: dict) -> list[list[str]]:
    return [decay["products"] for decays in decay_dict.values() for decay in decays]

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import product
from copy import deepcopy



def filter_channels_parallel(decay_dict, root, allowed_finals, max_workers=4):


    
    return 


In [53]:
decays22

{'D_s+': [{'branching_ratio': 0.0,
   'products': ['K+', 'K-', 'pi+'],
   'model': 'PHOTOS'},
  {'branching_ratio': 0.0, 'products': ['K+', 'K-', 'K+'], 'model': 'PHOTOS'}],
 'anti-B_s0': [{'branching_ratio': 0.0,
   'products': ['K-', 'K+', 'pi-'],
   'model': 'PHOTOS'},
  {'branching_ratio': 0.0, 'products': ['K-', 'K_S0'], 'model': 'PHOTOS'},
  {'branching_ratio': 0.0, 'products': ['D_s+', 'pi-'], 'model': 'PHOTOS'}]}

In [54]:
filtered1 = filter_channels_parallel(decays21, "B_s0", find_final_states("B_s0", decays1))
filtered2 = filter_channels_parallel(decays22, "anti-B_s0", find_final_states("anti-B_s0", decays1))


In [55]:
print(filtered1)
print(filtered2)


{'B_s0': [{'branching_ratio': 0.0, 'products': ['K+', 'K-', 'pi+'], 'model': 'PHOTOS'}, {'branching_ratio': 0.0, 'products': ['D_s-', 'pi+'], 'model': 'PHOTOS'}]}
{'anti-B_s0': [{'branching_ratio': 0.0, 'products': ['D_s+', 'pi-'], 'model': 'PHOTOS'}, {'branching_ratio': 0.0, 'products': ['K-', 'K+', 'pi-'], 'model': 'PHOTOS'}]}


In [155]:
find_final_states("B_s0", decays1)

[['K+', 'K-', 'pi+', 'pi-']]

In [49]:
d1 = {"aa": ["ab", "ba"], "asd" : ["saw",]}
d2 = {"aa": ["ab", "aswdvws"], }
d3 = dict()

In [50]:
merge_decay_dicts(d1, d2)

{'aa': ['aswdvws', 'ab', 'ba'], 'asd': ['saw']}

In [71]:
(5006578+5021813) - ((3542973+743255+30410+80059) + (3619246+790862+50212+79059))

1092315

10028391